# 03 · Filter & Rank — run the shared multi-layer filter (`design_type="antibody"`)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 15** you map your candidate variants onto `fp.Design` objects, run the pipeline with
the **antibody** cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12), and report survival
(D3 pt 1). The filter here enforces **pose maintenance** — a mutation that breaks the interface pose is
out, no matter how good its ESM-1v rank.

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"antibody"` cutoffs are the same as the rest of the antibody family (Projects 14–17).

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)
print("\nUsing design_type='antibody':", fp.DEFAULT_CUTOFFS["antibody"])

## Score the candidates' pose + developability, then build `fp.Design` objects

The campaign CSV has ESM-1v/AbLang **ranking** scores but not yet the **pose-maintenance** metrics
(those need the heavier AF2-Multimer step). We fill `pae_interaction` + `scrmsd` (and the developability
liability count) with `score_variants()` — on `tool="mock"` these are SYNTHETIC; on Colab switch to
`tool="af2"`. Then we map each variant onto an `fp.Design`: for an antibody complex the key fields are
`plddt`, `pae_interaction`, and `scrmsd`. We stash ESM-1v/AbLang/liabilities in `extra` so they ride
along into the ranked CSV.

In [ ]:
from maturation_tools import (Variant, score_variants, example_parent_sequence)

camp = pd.read_csv("results/campaign.csv")

# Rebuild lightweight Variant objects from the CSV to run the (heavier) pose check + dev scan.
# (On a real run you would already have AF2-Multimer outputs; here we compute the mock metrics.)
parent = example_parent_sequence()
variants = []
for _, r in camp.iterrows():
    muts = tuple(str(r["mutations"]).split("+")) if isinstance(r.get("mutations"), str) and r["mutations"] else ()
    # reconstruct the mutated sequence for pose/dev scoring (single + simple combos)
    seq = parent
    ok = True
    try:
        from maturation_tools import apply_mutation
        for m in muts:
            seq = apply_mutation(seq, m)
    except Exception:
        ok = False
    v = Variant(design_id=str(r["design_id"]), sequence=seq, antigen=str(r.get("antigen", "ANTIGEN")),
                mutations=muts, cdr=str(r.get("cdr", "")), source=str(r.get("source", "mock")),
                esm1v=r.get("esm1v"), ablang=r.get("ablang"))
    if ok:
        variants.append(v)
score_variants(variants, tool="mock")    # fills pae_interaction, scrmsd, n_liabilities (SYNTHETIC)
print(f"scored pose + developability for {len(variants)} variants (SYNTHETIC on mock)")

In [ ]:
designs = []
for v in variants:
    designs.append(fp.Design(
        design_id=v.design_id,
        sequence="",                      # full chain not needed for the confidence layers
        design_type="antibody",
        plddt=70.0 if v.pae_interaction is not None else None,  # mock proxy; real run -> AF2 interface pLDDT
        pae_interaction=v.pae_interaction,
        scrmsd=v.scrmsd,
        # solubility maps to a developability proxy so Layer 3 (physics) has something to act on;
        # fewer liabilities => "more soluble/developable" in this teaching mapping:
        solubility=(-float(v.n_liabilities) if v.n_liabilities is not None else None),
        extra={"esm1v": v.esm1v, "ablang": v.ablang, "n_liabilities": v.n_liabilities,
               "mutations": "+".join(v.mutations), "cdr": v.cdr,
               "synthetic": bool(v.synthetic)},
    ))
print(len(designs), "fp.Design objects built (design_type='antibody')")
print("NOTE: plddt here is a mock placeholder; on a real run use the AF2-Multimer interface pLDDT.")

## Run the pipeline + report

`run_pipeline(..., design_type="antibody")` applies the layers in order with the antibody cutoffs and
returns a ranked DataFrame; `report()` prints the **survival-at-each-layer** accounting and saves the
ranked CSV + figure. We run Layers 1 + 3 here (self-consistency/pose + physics/developability); Layer 2
(orthogonal predictor agreement) needs a second predictor — wire an ESMFold/IgFold scRMSD in for the
real run. **On mock data the survivors are SYNTHETIC** — the point is the plumbing and the honest
accounting.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="antibody", use_layers=(1, 3))
top = fp.report(df_ranked, top_n=15, save_prefix="results/proj15")
print("\nranked CSV -> results/proj15_ranked.csv ; survival figure -> results/proj15_survival.png")
top

## Hit-rate accounting (report the rate, not the cherry)

Survival = how many candidates keep the binding **pose** (and stay developable) after filtering. For
affinity maturation this is the honest, expected funnel: many scored mutations, far fewer that maintain
the pose, and a **small** final set to test. Crucially, passing the filter means "still binds in the
same pose and is developable" — **NOT** "binds tighter". Only SPR can confirm tighter binding.

In [ ]:
n_total = df_ranked.attrs.get("n_total", len(df_ranked))
survival = df_ranked.attrs.get("survival", {})
print(f"Candidates scored: {n_total}")
for layer, n in survival.items():
    print(f"  {layer}: {n} survivors ({100*n/max(n_total,1):.1f}%)")
print("\nlayers_passed distribution:")
if "layers_passed" in df_ranked:
    print(df_ranked["layers_passed"].value_counts().sort_index())
print("\nReminder: mock survivors are SYNTHETIC. Passing the filter = pose maintained + developable, "
      "NOT higher affinity. Only SPR (notebook 05) can confirm a real affinity gain.")

## D3 (part 1) checklist
- [ ] `results/proj15_ranked.csv` produced by the **shared** module with `design_type="antibody"`.
- [ ] Survival-at-each-layer reported (the survival figure saved); pose-maintenance enforced.
- [ ] Mapping assumptions written down (which metric → which `fp.Design` field; the plddt caveat).
- [ ] Honest hit-rate accounting; survivors framed as "pose-maintained + developable", not "tighter".

**Next:** `04_validate.ipynb` — pose maintenance, developability liability scan, epitope/epistasis.